# GWAS vs Linear Mixed Models: A Stress-Test Comparison Using 1000 Genomes chr22

This report summarizes the pipeline and findings from **main.ipynb**, which evaluates how genome-wide association (GWA) testing behaves under several deliberately challenging conditions (“stress scenarios”).

---

## Hypothesis

We hypothesize that **association test calibration (type I error and p-value validity) is sensitive to**:

1. **Population structure** — when the phenotype is correlated with ancestry (e.g., PC1), linear models without covariates may inflate test statistics.  
2. **Case–control imbalance** — binary traits with extreme case fractions may affect logistic regression stability and calibration.  
3. **Sample relatedness** — in highly related cohorts, linear mixed models (LMMs) may still show inflation if the phenotype aligns with genetic structure; in low-related cohorts, over-correction may cause deflation.  
4. **Genotype missingness** — randomly missing genotypes can bias effect estimates and p-values when not properly accounted for.

We test these by **simulating phenotypes** under controlled models and then running **PLINK (linear/logistic) and GEMMA (LMM)** association, and assessing **λGC and QQ plots** across scenarios.

---

## Introduction to Stress Scenarios (SS’s)

All phenotypes are simulated from **real chr22 genotypes** (QC’d, 97,216 SNPs; 2,504 individuals). The base model is:

\[
y = G_{\text{causal}} \beta + C + \epsilon
\]

where \(G_{\text{causal}}\) is a set of randomly chosen causal SNPs, \(\beta\) are random effects, \(C\) is a “stress” component, and \(\epsilon\) is Gaussian noise. We define four stress scenarios:

| SS | Name | Description |
|----|------|-------------|
| **SS1** | Population structure confounding | \(C = a \cdot \text{PC1}\); we vary \(a \in \{0, 0.5, 1, 2, 3\}\) to inject ancestry-related signal. |
| **SS2** | Case–control imbalance | Binary trait: \(\text{logit}(P(\text{case})) = G\beta + \alpha\); we vary \(\alpha\) to get case proportions from ~34% to ~99%. |
| **SS3** | Sample relatedness stress | Two cohorts: **high-related** (200 individuals with strongest kinship) and **low-related** (200 with minimal kinship). Phenotype \(y = \text{PC1} + \epsilon\); chr22 LMM run per cohort. |
| **SS4** | Genotype missingness | Randomly mask a fraction of genotype entries (e.g., 1%, 5%, 10%, 20%), then re-run simple linear association (y ~ SNP) and compare p-value distributions. |

Baseline (no stress): \(C = 0\), continuous \(y = G_{\text{causal}}\beta + \epsilon\); used for PLINK linear (with/without PC covariates) and for comparison with stressed runs.

---

## Methods

### Data and preprocessing (main.ipynb §§1–3)

- **Genotypes:** 1000 Genomes–style VCFs for chr20–22; converted to PLINK bed/bim/fam. QC: MAF ≥ 0.05, geno ≤ 0.02, HWE \(p \leq 10^{-6}\). LD pruning (50/5/0.2) for PCA and GRM.
- **Genome-wide dataset (chr20–22):** Used for PCA (10 PCs), covariates file (FID, IID, PC1–PC5), and GRM (GEMMA `-gk 1`); ~21k LD-pruned SNPs, 2,504 individuals.
- **chr22-only dataset:** Used for all association tests; 97,216 SNPs after QC.

### Phenotype simulation (§4)

- **SS1:** Continuous \(y = G\beta + a\cdot\text{PC1} + \epsilon\); outputs in `output/pheno_pc_confound/`.
- **SS2:** Binary from logistic model; outputs in `output/pheno_casecontrol/`.
- **SS3:** Phenotype = PC1 + small noise for highrel/lowrel subsets; GRM and LMM run per cohort.
- **SS4:** Same baseline phenotype; genotypes masked at given missing rates; per-SNP linear regression, results in `output/assoc_missingness/`.

### Association testing (§5)

- **PLINK:** Linear (with/without `covariates.txt`) on continuous baseline; logistic on binary phenotype. Outputs: `plink_nocov`, `plink_pc`, `plink_logistic`.
- **GEMMA:** LMM (`-lmm 4`) on chr22 for high-related and low-related cohorts, with cohort-specific GRM.

### Calibration and visualization (§6)

- **λGC:** \(\lambda_{GC} = \text{median}(\chi^2_{1}) / 0.456\) over association p-values.
- **QQ plots:** Observed vs expected \(-\log_{10}(p)\) for each result file (optional Manhattan/summary in main.ipynb).
- **Summary:** λGC table; significant-hit and top-K overlap between PLINK and GEMMA where applicable.

---

## Results

### Baseline and SS1 (population structure)

- **PLINK linear, no covariates:** Strong inflation (λGC ≫ 1) when phenotype is correlated with structure.
- **PLINK linear, with PC covariates:** λGC near 1 for baseline and moderate \(a\); adding PCs restores calibration.
- **SS1 grid:** As \(a\) increases, inflation in the no-covariate model increases; with covariates, calibration is largely restored.

### SS2 (case–control imbalance)

- Binary phenotypes with case proportion from ~34% to ~99% were generated. PLINK logistic association was run; calibration can be compared across \(\alpha\) (e.g., via λGC or QQ) to assess robustness to imbalance.

### SS3 (sample relatedness)

- **High-related cohort (n=200):** λGC ≈ 1.27 — moderate inflation; QQ plot shows upward deviation. Phenotype (PC1 + noise) aligns with genetic structure; LMM partially but not fully corrects.
- **Low-related cohort (n=200):** λGC ≈ 0.78 — deflation; QQ near or below the null line. Suggests over-correction when kinship is weak.

### SS4 (genotype missingness)

- Association was re-run with 1%, 5%, 10%, and 20% of genotypes randomly set to missing. Per-SNP regression p-values were saved per rate. Higher missingness can induce bias and inflation; λGC and QQ can be compared across rates (see `output/assoc_missingness/`).

### Method agreement (§6.3)

- For baseline, PLINK (with PCs) and GEMMA show much better agreement (e.g., Jaccard on top hits) than PLINK without covariates, consistent with confounding in the no-covariate case.

---

## Interpretation

- **Population structure (SS1):** Confirms that uncorrected linear models inflate when the trait tracks ancestry; including PC covariates restores calibration.
- **Case–control imbalance (SS2):** Illustrates that logistic regression is applied across a range of case fractions; calibration metrics can be used to flag problematic imbalance.
- **Relatedness (SS3):** LMM calibration depends on the match between relatedness and phenotype structure: high relatedness + structured phenotype → residual inflation; low relatedness → possible deflation from over-correction.
- **Missingness (SS4):** Random missingness can distort p-values; the pipeline provides a way to quantify this via λGC and QQ across missingness rates.

Together, the four stress scenarios and baseline analyses in **main.ipynb** provide a clear picture of when and how GWA calibration degrades, and how covariates and LMMs can mitigate or exacerbate it.

---

## Summary

| Scenario | Main idea | Key output / location |
|----------|-----------|------------------------|
| **Baseline** | Continuous trait, no stress | `output/pheno_baseline.txt`; PLINK linear ± PCs |
| **SS1** | PC confounding (\(a \cdot \text{PC1}\)) | `output/pheno_pc_confound/`; compare λGC with/without covariates |
| **SS2** | Case–control imbalance (varying \(\alpha\)) | `output/pheno_casecontrol/`; PLINK logistic |
| **SS3** | High vs low relatedness (n=200 each) | `output/assoc_highrel.assoc.txt`, `assoc_lowrel.assoc.txt`; λGC ≈ 1.27 vs 0.78 |
| **SS4** | Random genotype missingness (1–20%) | `output/assoc_missingness/assoc_missing_rate*.tsv` |

All pipelines, plots (QQ, λGC table, overlap statistics), and exact commands are in **main.ipynb**; this report states the hypothesis, introduces the SS’s, and summarizes methods and results.